# CANOPy Sensor Position Reconstruction Tutorial

Welcome to the Sensor Position Reconstruction tutorial! This notebook will guide you through the process of
reconstructing sensor positions solely from a point cloud and the tools provided in the CANOPy toolbox. Improvements implemented towards [Gassilloud et al. (2025)](https://www.sciencedirect.com/science/article/pii/S1569843225001402) are noted [here](../README.md#changelog). Licensed under the [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/).

<br>


### Why sensor position reconstruction?

Sensor position reconstruction is usefull, when the LiDAR scanners position is not known or not adequatly represented by airborne vehicle trajectories (i.e. the LiDAR sensor is mounted on a moving gimbal). Applications such as [occlusion mapping](../../occlusion_mapping/) and transmissivity metrics rely on ray tracing algorithms, which require precise sensor positions for beam trajectory reconstruction.

<br>


### Installation

Follow the installation instructions [here](../../README.md#installation).

<br>


### Example Data

In this tutorial, we will use example data and configuration parameters provided in `./example`. **Download example data** [here](https://www.bwsyncandshare.kit.edu/s/nS9xAFG6qBdqf7T) and move the data to `./example/data`. Adapt the corresponding file paths in the [example config file](./example_config_sensor_position_reconstruction.yml).

<br>


**Let's get started!**




<br>

## Sensor Position Reconstruction

In LiDAR scanning, the sensor emits laser pulses and records the returns. By reconstructing pulse trajectories with multiple returns and extending the trajectory backwards, we can estimate the sensor's position. We have some parameters already defined in our [example config file](./example_config_sensor_position_reconstruction.yml).


The main steps are:

1. Reconstruct trajectories with multiple returns
2. Calculate closest points between consecutive trajectories
3. Create time intervals, calculate median of closest points -> this is our sensor position
4. Assign gps_time of closest trajectory to sensor position

Returns: sensor_position_reconstruction.gpkg

<br>


#### Option 1: Command Line

To run sensor position reconstruction from the command line provide the correct paths (also in the config file):

```bash
python path/to/sensor_position_reconstruction.py --config path/to/example_config_sensor_position_reconstruction.yml
```


#### Option 2: Python

Now, let's reconstruct sensor positions using the python modules. We'll use directly the function `reconstruct_uav_sensor_trajectory` but feel free to take a closer look at the available functions.

In [1]:
import sys
import os
from pathlib import Path

current_dir = Path(os.getcwd()).parent.parent.parent
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

from CANOPy.sensor_position_reconstruction.config import create_sensor_position_reconstruction_config
from CANOPy.sensor_position_reconstruction.sensor_position_reconstruction import reconstruct_uav_sensor_trajectory

# create config
config_path = Path("./example_config_sensor_position_reconstruction.yml")  # path to example config file
cfg = create_sensor_position_reconstruction_config(config_path)  # create config

# read parameters
point_cloud_path = cfg["point_cloud_path"]
epsg = cfg["epsg_code"]
sensor_position_path = cfg["position_reconstruction"]
extension_max = cfg["sensor_position_reconstruction_kwargs"]["extension_max"]
positions_per_second = cfg["sensor_position_reconstruction_kwargs"]["positions_per_second"]
traj_number_min = cfg["sensor_position_reconstruction_kwargs"]["traj_number_min"]
distance_max = cfg["sensor_position_reconstruction_kwargs"]["distance_max"]
extrapolate = cfg["sensor_position_reconstruction_kwargs"]["extrapolate"]

# run trajectory reconstruction
reconstruct_uav_sensor_trajectory(point_cloud_path=point_cloud_path,
                                  sensor_position_path=sensor_position_path,
                                  epsg=epsg,
                                  extension_max=extension_max,
                                  positions_per_second=positions_per_second,
                                  traj_number_min=traj_number_min,
                                  distance_max=distance_max,
                                  extrapolate=extrapolate)

Read point cloud.
    read all points.
    read dimensions: ['gps_time', 'x', 'y', 'z', 'return_number']
    metadata: {'Points Read': 10488086, 'Total point_count': 10488086, 'gps_encoding_value': 0, 'gps_time_type': <GpsTimeType.WEEK_TIME: 0>, 'dimensions': ['gps_time', 'x', 'y', 'z', 'return_number'], 'las_header': <LasHeader(1.2, <PointFormat(3, 0 bytes of extra dims)>)>}
Sort point cloud, count unique values
Extend pulses with multiple returns
Reconstruct sensor position from closest points
Reconstructed 3175 positions
Assign gps_time of closest trajectories to reconstructed sensor positions.
Average number of trajectories for each reconstructed sensor position: 696.3914960629921
Average distance between closest trajectory and sensor position: 0.00024610108292964826
Assigned gps_time to 3175 positions
Extrapolate to start position.
Extrapolate to last position.
Store file.
Sensor reconstruction complete.


<br>


In result, 3175 sensor positions could be reconstructed. We can see, that the distances between a reconstructed sensor position and their closest reconstructed LiDAR beam trajectory from multiple returns are smaller the `distance_max` value set in our config file.

![Distance between reconstructed sensor position and closest trajectory for time assignment.](./sensor_position_reconstruction/min_dist.png "Reconstructed sensor positions.")

<br>


Therefore, we were able to assign `gps_time` from the closest beam trajectory to all our sensor positions. The resulting sensor positions are exported as a .gpkg file. Lets have a look at the results:

![Reconstructed sensor positions.](./sensor_position_reconstruction/sensor_position_example1.png "Reconstructed sensor positions.")

<br>


We can see that the reconstructed sensor positions represent the movement trajectroy of our LiDAR sensor. Lets take a closer look:

![Reconstructed sensor positions, close.](./sensor_position_reconstruction/sensor_position_example2.png "Reconstructed sensor positions, close.")


<br>

### Conclusion

This tutorial demonstrates the reconstruction of sensor positions from a point cloud.

For a detailled explanation have a look at [Gassilloud, M., Koch, B., & Goeritz, A. (2025)](https://www.sciencedirect.com/science/article/pii/S1569843225001402) Occlusion mapping reveals the impact of flight and sensing parameters on vertical forest structure exploration with cost-effective UAV based laser scanning. International Journal of Applied Earth Observation and Geoinformation, 139, 104493.